In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

CATALOG = "smart_claims_sesion_5"
BRONZE_SCHEMA = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/landing/data"

CUSTOMERS_PATH = f"{VOLUME_PATH}/customers/customers.csv"
POLICIES_PATH = f"{VOLUME_PATH}/policies/policies.csv"
CLAIMS_PATH = f"{VOLUME_PATH}/claims/claims.csv"
TELEMATICS_PATH = f"{VOLUME_PATH}/telematics/"
CLAIM_IMAGES_PATH = f"{VOLUME_PATH}/claims_images/"
TRAINING_IMGS_PATH = f"{VOLUME_PATH}/training_imgs/"
IMAGE_METADATA_PATH = f"{VOLUME_PATH}/claim_images_metadata/image_metadata.csv"

spark = SparkSession.builder.appName("smart_claims").getOrCreate()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

# customers
(spark.read.option("header", "true").option("inferSchema", "true")
    .csv(CUSTOMERS_PATH)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers"))
print("customers cargado")

# policies
(spark.read.option("header", "true").option("inferSchema", "true")
    .csv(POLICIES_PATH)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.policies"))
print("policies cargado")

# claims
(spark.read.option("header", "true").option("inferSchema", "true")
    .csv(CLAIMS_PATH)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.claims"))
print("claims cargado")

# telematics
(spark.read.parquet(TELEMATICS_PATH)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.telematics"))
print("telematics cargado")

# training images
(spark.read.format("binaryFile").option("recursiveFileLookup", "true")
    .load(TRAINING_IMGS_PATH)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.training_images"))
print("training_images cargado")

# claim images
(spark.read.format("binaryFile").option("recursiveFileLookup", "true")
    .load(CLAIM_IMAGES_PATH)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images"))
print("claim_images cargado")

# image metadata
(spark.read.option("header", "true").option("inferSchema", "true")
    .csv(IMAGE_METADATA_PATH)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.image_metadata"))
print("image_metadata cargado")

print("Bronze lista")

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}"))

In [0]:
tablas = ["customers", "policies", "claims", "telematics", "training_images", "claim_images", "image_metadata"]

for tabla in tablas:
    print(f"\n{tabla}")
    print(f"registros: {spark.sql(f'SELECT COUNT(*) FROM {CATALOG}.{BRONZE_SCHEMA}.{tabla}').collect()[0][0]}")
    spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.{tabla} LIMIT 2").show(truncate=False)